# `expansion_hunter_02_run_batch` — native Google Batch

Run **in a VWB JupyterLab app** after
[`expansion_hunter_00_prep_reference.ipynb`](expansion_hunter_00_prep_reference.ipynb).

This submits **minicram** then **genotype** with `gcloud batch`. No dsub
sidecars. Proven on sample `1000000`: two-locus catalog
(`eh-mini-260929-035922`, `eh-gt-260929-041339`) and the 711-locus
degenerate panel (`eh-mini-260929-051752`, `eh-gt-260929-053643`; 67.5 MiB
minicram, ~8 min minicram + ~1 min genotype).

**Production catalog** is that degenerate panel
[`candidate_EH_Loci.GRCh38.degenerate.json`](../../expansion_hunter/configs/candidate_EH_Loci.GRCh38.degenerate.json)
(`CATALOG_KIND = "degenerate"`). The two-locus extract is not reusable for
it. Set `CATALOG_KIND = "tiny"` only to replay the two-locus path.

This notebook is the **one-sample smoke**. Cohort cost + keep list +
sharded dispatch live in
[`expansion_hunter_03_budget.ipynb`](expansion_hunter_03_budget.ipynb) and
[`expansion_hunter_04_dispatch.ipynb`](expansion_hunter_04_dispatch.ipynb).
Status of every Batch job: [`batch_monitor.ipynb`](batch_monitor.ipynb).

The WDL notebook ([`expansion_hunter_01_run.ipynb`](expansion_hunter_01_run.ipynb))
is unchanged. Docs: [`expansion_hunter/batch/README.md`](../../expansion_hunter/batch/README.md).

## What this notebook does

1. Check `PET_SA_EMAIL`, `GOOGLE_CLOUD_PROJECT`, `gcloud`
2. Stage the chosen catalog JSON to GCS if it is missing
3. Write a per-sample CSV (same columns as `configs/batch.header.csv`)
4. Build Batch job JSON for minicram and genotype (dry-run; always)
5. Optionally submit one stage (`SUBMIT_MINICRAM` or `SUBMIT_GENOTYPE`)
6. Describe the job and list GCS outputs

`SUBMIT_*` stay **off** until you turn one on. Submit genotype only after
minicram objects exist for that sample **and this catalog**.

The WGS CRAM stays a `gs://` env var so Batch never localizes the Nearline
object.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        script = p / "expansion_hunter" / "batch" / "submit_batch.py"
        if script.is_file():
            return p
        nested = p / "aou-lr-phase-2" / "expansion_hunter" / "batch" / "submit_batch.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError(
        "Cannot find expansion_hunter/batch/submit_batch.py. Clone "
        "kvg/aou-lr-phase-2 into this app (or cd into the clone) and re-run."
    )


def sh(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check, text=True)


def gcs_exists(uri: str) -> bool:
    return sh(["gcloud", "storage", "ls", uri], check=False).returncode == 0


def summarize_job(label: str, job: dict, cfg_path: Path) -> None:
    group = job["taskGroups"][0]
    spec = group["taskSpec"]
    env = dict(spec["environment"]["variables"])
    runnable = spec["runnables"][1] if "script" in spec["runnables"][0] else spec["runnables"][0]
    container = runnable["container"]
    print(f"=== {label} ===")
    print("config:", cfg_path)
    print("tasks:", group["taskCount"])
    print("runnables:", len(spec["runnables"]))
    print("image:", container["imageUri"])
    print("entrypoint:", container["entrypoint"])
    print("cpuMilli:", spec["computeResource"]["cpuMilli"], "memoryMib:", spec["computeResource"]["memoryMib"])
    print("bootDiskMib:", spec["computeResource"]["bootDiskMib"])
    print("maxRunDuration:", spec["maxRunDuration"])
    print("SAMPLE_ID:", env.get("SAMPLE_ID"))
    for key in ("CRAM", "CATALOG", "MINICRAM", "EH_JSON", "EH_VCF", "WORKER_LOG_GCS"):
        if key in env:
            print(f"{key}:", env[key])
    if "taskEnvironments" in group:
        print("per-task samples:", [e["variables"]["SAMPLE_ID"] for e in group["taskEnvironments"]])


REPO_ROOT = find_repo_root()
BATCH_DIR = REPO_ROOT / "expansion_hunter" / "batch"
CONFIGS_DIR = REPO_ROOT / "expansion_hunter" / "configs"
MINICRAM_PY = BATCH_DIR / "make_minicram.py"
GENOTYPE_PY = BATCH_DIR / "genotype.py"
TINY_CATALOG_LOCAL = CONFIGS_DIR / "smoke.catalog.json"
DEG_CATALOG_LOCAL = CONFIGS_DIR / "candidate_EH_Loci.GRCh38.degenerate.json"
SCRATCH = Path.cwd() / "expansion_hunter_rw_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BATCH_DIR))
from submit_batch import build_genotype_job, build_minicram_job  # noqa: E402

OUTPUT_BUCKET_GS = os.environ.get("EH_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")
REGION = os.environ.get("DSUB_REGION", "us-central1")

SAMPLE_ID = os.environ.get("EH_SAMPLE_ID", "1000000")
CRAM = os.environ.get(
    "EH_CRAM",
    "gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram",
)
CRAI = os.environ.get(
    "EH_CRAI",
    "gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram.crai",
)
REF_FA = os.environ.get(
    "EH_REF_FA",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta",
)
REF_FAI = os.environ.get(
    "EH_REF_FAI",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta.fai",
)
SEX = os.environ.get("EH_SEX", "female")
GCLOUD_PROJECT = os.environ.get("EH_GCLOUD_PROJECT", os.environ.get("GOOGLE_CLOUD_PROJECT", ""))

PRINT_READS_DOCKER = os.environ.get(
    "EH_PRINT_READS_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-locityper-print-reads:0.1.2",
)
EH_DOCKER = os.environ.get(
    "EH_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-expansion-hunter:0.1.0",
)

# "tiny" = 2-locus smoke (already SUCCEEDED). "degenerate" = 711-locus production panel.
CATALOG_KIND = os.environ.get("EH_CATALOG_KIND", "degenerate")
STAGE_CATALOG = True  # copy local JSON to GCS if missing; does not submit Batch
BATCH_CSV_LOCAL = os.environ.get("EH_BATCH_CSV", "")

SUBMIT_MINICRAM = False
SUBMIT_GENOTYPE = False
WATCH_JOB_ID = ""  # or leave empty to describe last submitted ids
JOB_ID_PATH = SCRATCH / "last_minicram.job_id"
GT_JOB_ID_PATH = SCRATCH / "last_genotype.job_id"

CATALOG_PREFIX = f"{OUTPUT_BUCKET_GS.rstrip('/')}/expansion_hunter"
TINY_CATALOG_GS = f"{CATALOG_PREFIX}/smoke.catalog.json"
DEG_CATALOG_GS = f"{CATALOG_PREFIX}/candidate_EH_Loci.GRCh38.degenerate.json"
if CATALOG_KIND == "tiny":
    CATALOG_LOCAL = TINY_CATALOG_LOCAL
    CATALOG_DEFAULT = TINY_CATALOG_GS
    OUT_PREFIX = os.environ.get(
        "EH_BATCH_OUT_PREFIX",
        f"{OUTPUT_BUCKET_GS.rstrip('/')}/batchRuns/expansion_hunter/tiny",
    )
    MAX_RUN_DURATION = "14400s"
elif CATALOG_KIND == "degenerate":
    CATALOG_LOCAL = DEG_CATALOG_LOCAL
    CATALOG_DEFAULT = DEG_CATALOG_GS
    OUT_PREFIX = os.environ.get(
        "EH_BATCH_OUT_PREFIX",
        f"{OUTPUT_BUCKET_GS.rstrip('/')}/batchRuns/expansion_hunter",
    )
    MAX_RUN_DURATION = "28800s"
else:
    raise SystemExit(f"CATALOG_KIND must be 'tiny' or 'degenerate', got {CATALOG_KIND!r}")

CATALOG = os.environ.get("EH_CATALOG", CATALOG_DEFAULT)

print("REPO_ROOT:", REPO_ROOT)
print("MINICRAM_PY:", MINICRAM_PY, "exists=" + str(MINICRAM_PY.is_file()))
print("GENOTYPE_PY:", GENOTYPE_PY, "exists=" + str(GENOTYPE_PY.is_file()))
print("CATALOG_KIND:", CATALOG_KIND)
print("CATALOG_LOCAL:", CATALOG_LOCAL, "exists=" + str(CATALOG_LOCAL.is_file()))
if CATALOG_LOCAL.is_file():
    recs = json.loads(CATALOG_LOCAL.read_text())
    print("catalog records:", len(recs) if isinstance(recs, list) else type(recs).__name__)
print("CATALOG:", CATALOG)
print("OUT_PREFIX:", OUT_PREFIX)
print("MAX_RUN_DURATION:", MAX_RUN_DURATION)
print("SAMPLE_ID:", SAMPLE_ID or "(unset)")
print("CRAM:", CRAM)
print("SEX:", SEX)
print("PRINT_READS_DOCKER:", PRINT_READS_DOCKER)
print("EH_DOCKER:", EH_DOCKER)
print("GCLOUD_PROJECT:", GCLOUD_PROJECT or "(unset)")
print("PET_SA_EMAIL:", os.environ.get("PET_SA_EMAIL", "(unset)"))
print("STAGE_CATALOG:", STAGE_CATALOG)
print("SUBMIT_MINICRAM:", SUBMIT_MINICRAM, "SUBMIT_GENOTYPE:", SUBMIT_GENOTYPE)
print("gcloud:", shutil.which("gcloud"))


## PET SA / VPC

Jobs run as `PET_SA_EMAIL` on the workspace private VPC (`network` /
`subnetwork` in `us-central1`). No `dsub`. PET cannot read Cloud Logging;
after SUCCESS read `{sample}.minicram.worker.log` / `{sample}.EH.worker.log`.


In [ ]:
if shutil.which("gcloud") is None:
    raise SystemExit("gcloud is not on PATH. Open this notebook in a VWB Jupyter app.")
if not os.environ.get("GOOGLE_CLOUD_PROJECT"):
    raise SystemExit("GOOGLE_CLOUD_PROJECT is empty. Open this notebook in a VWB Jupyter app.")
if not os.environ.get("PET_SA_EMAIL"):
    raise SystemExit("PET_SA_EMAIL is empty. VWB sets this in cloud apps.")
if not MINICRAM_PY.is_file():
    raise SystemExit(f"missing worker: {MINICRAM_PY}")
if not GENOTYPE_PY.is_file():
    raise SystemExit(f"missing worker: {GENOTYPE_PY}")
if not CATALOG_LOCAL.is_file():
    raise SystemExit(f"missing catalog: {CATALOG_LOCAL}")

sh(["gcloud", "config", "get-value", "project"], check=False)
print("project:", os.environ["GOOGLE_CLOUD_PROJECT"])
print("PET_SA_EMAIL:", os.environ["PET_SA_EMAIL"])
print("network: projects/{}/global/networks/network".format(os.environ["GOOGLE_CLOUD_PROJECT"]))
print("subnetwork: projects/{}/regions/{}/subnetworks/subnetwork".format(
    os.environ["GOOGLE_CLOUD_PROJECT"], REGION
))


## Stage catalog to GCS

Copies the chosen local JSON to `CATALOG` if that object is missing. Does
not submit Batch. The two-locus smoke is already on GCS; the degenerate
panel needs this once before the 711-locus minicram.


In [ ]:
recs = json.loads(CATALOG_LOCAL.read_text())
print(CATALOG_LOCAL.name, "records", len(recs) if isinstance(recs, list) else type(recs).__name__)
if not STAGE_CATALOG:
    print("STAGE_CATALOG=False; skip copy")
    print("CATALOG:", CATALOG)
elif gcs_exists(CATALOG):
    print("skip upload; exists", CATALOG)
else:
    sh(["gcloud", "storage", "cp", str(CATALOG_LOCAL), CATALOG])
    print("staged catalog:", CATALOG)


## Batch CSV

Same columns as [`expansion_hunter/configs/batch.header.csv`](../../expansion_hunter/configs/batch.header.csv).
The WGS CRAM **and** CRAI stay `gs://` (never localized). This cell always
runs; it does not submit.


In [ ]:
csv_path = Path(BATCH_CSV_LOCAL) if BATCH_CSV_LOCAL else SCRATCH / "expansion_hunter.batch.csv"
if BATCH_CSV_LOCAL:
    if not csv_path.is_file():
        raise SystemExit(f"BATCH_CSV_LOCAL does not exist: {csv_path}")
    print("using existing CSV:", csv_path)
else:
    required = {
        "SAMPLE_ID": SAMPLE_ID,
        "CRAM": CRAM,
        "CRAI": CRAI,
        "REF_FA": REF_FA,
        "REF_FAI": REF_FAI,
        "CATALOG": CATALOG,
    }
    missing = [k for k, v in required.items() if not v]
    if missing:
        raise SystemExit(f"fill {missing} in the config cell (or set BATCH_CSV_LOCAL)")
    fieldnames = ["sample_id", "cram", "crai", "ref_fa", "ref_fai", "catalog", "sex", "gcloud_project"]
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        w.writerow(
            {
                "sample_id": SAMPLE_ID,
                "cram": CRAM,
                "crai": CRAI,
                "ref_fa": REF_FA,
                "ref_fai": REF_FAI,
                "catalog": CATALOG,
                "sex": SEX,
                "gcloud_project": GCLOUD_PROJECT,
            }
        )
    print("wrote one-row CSV:", csv_path)

print(csv_path.read_text())


## Build minicram JSON (dry-run)

Always runs. Does not submit. The worker is embedded as `python3 -c …`.
Confirm `CATALOG` is the degenerate URI and `maxRunDuration` is 8h before
submitting the 711-locus extract.


In [ ]:
project = os.environ["GOOGLE_CLOUD_PROJECT"]
sa = os.environ["PET_SA_EMAIL"]
mini_job = build_minicram_job(
    csv_path=csv_path,
    worker_path=MINICRAM_PY,
    image=PRINT_READS_DOCKER,
    out_prefix=OUT_PREFIX,
    project=project,
    region=REGION,
    sa=sa,
    max_run_duration=MAX_RUN_DURATION,
)
mini_cfg = SCRATCH / "minicram.batch.json"
mini_cfg.write_text(json.dumps(mini_job, indent=2) + "\n", encoding="utf-8")
summarize_job("minicram", mini_job, mini_cfg)


## Submit minicram

Set **only** `SUBMIT_MINICRAM = True` in the config cell and re-run from
there. The two-locus job `eh-mini-260929-035922` does **not** count for
`CATALOG_KIND = "degenerate"` — that extract is 1.5 MiB and will be
overwritten under the production prefix.


In [ ]:
n_flags = int(SUBMIT_MINICRAM) + int(SUBMIT_GENOTYPE)
if n_flags > 1:
    raise SystemExit("set only one of SUBMIT_MINICRAM, SUBMIT_GENOTYPE")

if not SUBMIT_MINICRAM:
    print("SUBMIT_MINICRAM is off; not submitting.")
else:
    job_id = "eh-mini-" + datetime.now(timezone.utc).strftime("%y%m%d-%H%M%S")
    sh(
        [
            "gcloud",
            "batch",
            "jobs",
            "submit",
            job_id,
            "--location",
            REGION,
            "--project",
            os.environ["GOOGLE_CLOUD_PROJECT"],
            "--config",
            str(mini_cfg),
        ]
    )
    JOB_ID_PATH.write_text(job_id + "\n", encoding="utf-8")
    print("JOB_ID=" + job_id)
    print("wrote", JOB_ID_PATH)


## Build genotype JSON (dry-run)

Always runs. Does not submit. Host runnables copy minicram + fasta with
`gcloud storage`; the EH container only runs ExpansionHunter.


In [ ]:
gt_job = build_genotype_job(
    csv_path=csv_path,
    worker_path=GENOTYPE_PY,
    image=EH_DOCKER,
    out_prefix=OUT_PREFIX,
    project=project,
    region=REGION,
    sa=sa,
    max_run_duration=MAX_RUN_DURATION,
)
gt_cfg = SCRATCH / "genotype.batch.json"
gt_cfg.write_text(json.dumps(gt_job, indent=2) + "\n", encoding="utf-8")
summarize_job("genotype", gt_job, gt_cfg)


## Submit genotype

Requires the minicram objects for this sample **and this catalog**. Set
**only** `SUBMIT_GENOTYPE = True` in the config cell and re-run from there.


In [ ]:
if not SUBMIT_GENOTYPE:
    print("SUBMIT_GENOTYPE is off; not submitting.")
else:
    sample_cram = f"{OUT_PREFIX.rstrip('/')}/{SAMPLE_ID}/{SAMPLE_ID}.minicram.cram"
    print("checking", sample_cram)
    ls = sh(["gcloud", "storage", "ls", sample_cram], check=False)
    if ls.returncode != 0:
        raise SystemExit("minicram object missing; wait for minicram SUCCEEDED first")
    job_id = "eh-gt-" + datetime.now(timezone.utc).strftime("%y%m%d-%H%M%S")
    sh(
        [
            "gcloud",
            "batch",
            "jobs",
            "submit",
            job_id,
            "--location",
            REGION,
            "--project",
            os.environ["GOOGLE_CLOUD_PROJECT"],
            "--config",
            str(gt_cfg),
        ]
    )
    GT_JOB_ID_PATH.write_text(job_id + "\n", encoding="utf-8")
    print("JOB_ID=" + job_id)
    print("wrote", GT_JOB_ID_PATH)


## Status / outputs

Set `WATCH_JOB_ID` to a specific id, or this cell describes the last
minicram and/or genotype ids it wrote. PET cannot read Cloud Logging.


In [ ]:
ids: list[tuple[str, str]] = []
if (WATCH_JOB_ID or "").strip():
    ids.append(("watch", WATCH_JOB_ID.strip()))
else:
    if JOB_ID_PATH.is_file():
        ids.append(("minicram", JOB_ID_PATH.read_text(encoding="utf-8").strip()))
    if GT_JOB_ID_PATH.is_file():
        ids.append(("genotype", GT_JOB_ID_PATH.read_text(encoding="utf-8").strip()))

print("CATALOG_KIND:", CATALOG_KIND)
print("CATALOG:", CATALOG)
print("OUT_PREFIX:", OUT_PREFIX)
print("expected outputs:")
prefix = f"{OUT_PREFIX.rstrip('/')}/{SAMPLE_ID}/{SAMPLE_ID}"
for suffix in (
    ".minicram.cram",
    ".minicram.cram.crai",
    ".data_transfer_stats.tsv",
    ".minicram.worker.log",
    ".EH.json",
    ".EH.vcf",
    ".EH.worker.log",
):
    print(" ", prefix + suffix)

if not ids:
    print("no JOB_ID yet (submit, or set WATCH_JOB_ID)")
else:
    for label, job_id in ids:
        print(f"=== {label} {job_id} ===")
        sh(
            [
                "gcloud",
                "batch",
                "jobs",
                "describe",
                job_id,
                "--location",
                REGION,
                "--project",
                os.environ["GOOGLE_CLOUD_PROJECT"],
                "--format=yaml(status.state,status.statusEvents)",
            ],
            check=False,
        )

sample_dir = f"{OUT_PREFIX.rstrip('/')}/{SAMPLE_ID}/"
print("listing", sample_dir)
sh(["gcloud", "storage", "ls", "-l", sample_dir], check=False)
